In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- TPU guard: CLI pushes get a plain CPU machine; the real run is started from the editor with
#     Accelerator = TPU v5e-8. Stop at once if there is no TPU (checked in a subprocess so this
#     notebook never holds the chips that the training processes need).
os.environ.pop("TPU_PROCESS_ADDRESSES", None)
probe = subprocess.run([sys.executable, "-c", "import torch_xla, torch_xla.core.xla_model as xm; "
                        "print(xm.xla_device_hw(torch_xla.device()))"], capture_output=True, text=True)
print("accelerator:", probe.stdout.strip() or probe.stderr.strip()[-300:])
if "TPU" not in probe.stdout:
    raise SystemExit("No TPU. Open this notebook in the Kaggle editor, Accelerator -> TPU v5e-8, "
                     "then Save Version -> Save & Run All.")


In [ ]:
# --- locate the tokenizer and all token-shard dirs among the attached kernel outputs ---
DATA_DIRS = inputs_with("manifest.json", "vishalam-data-")
TOKENIZER = inputs_with("tokenizer.json", "vishalam-data-1")[0] + "/tokenizer.json"
print("shard dirs:", DATA_DIRS)


In [ ]:
# --- Pretraining session on TPU v5e-8 (8 chips, bf16): resume from the latest 350m session
#     (GPU or TPU, checkpoints are interchangeable), stop cleanly before Kaggle's 9 h TPU limit ---
prev = inputs_with("run/ckpt.pt", "vishalam-pretrain-350m-")
resume = f"--resume-from {prev[0]}/run" if prev else ""
print("resuming from:", prev or "scratch")
cmd = (f"python -m vishalam.train.pretrain --xla --config configs/vishalam-350m.yaml "
       f"--data-dirs {','.join(DATA_DIRS)} --tokenizer {TOKENIZER} {resume} --out {WORK}/run "
       f"--max-hours {hours_left(8.3):.2f} --set micro_batch=4 --set grad_checkpointing=true")
# watchdog: a stuck run (e.g. a collective that never completes) would otherwise burn the whole
# 9 h session; no output for 40 min (first-step compile is ~1 min) -> stop it
import threading
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
last = [time.time()]
def watch():
    while proc.poll() is None:
        if time.time() - last[0] > 2400:
            print("WATCHDOG: no output for 40 min - stopping the run", flush=True)
            proc.kill()
        time.sleep(30)
threading.Thread(target=watch, daemon=True).start()
for line in proc.stdout:
    last[0] = time.time()
    if "Warning" not in line:
        print(line, end="", flush=True)
print("exit code", proc.wait())
